# §40: EDA of the Nexar training corpus K5f reads, `Nexar_window` (CPU only, no training)

Rule: `core/docs/final/KIM_RUN.md` **§40**. Dataset properties only; nothing gates and no verdict changes. K5f and K5dfi read the **same** corpus (DVS is a training-time splice, not a corpus change), so one report serves both.

| step | what | GPU | estimate |
|---|---|---|---|
| 1-2 | mount, setup + preflight | no | 1-2 min |
| 3 | pytest on the tool - **raises if red** | no | < 1 min |
| 4 | corpus gate (7,311 windows = 1,689 + 5,622, positive videos only) | no | < 1 min |
| 5 | `core.tools.nexar_eda --plots` (the rebuilt labels are gated against `meta.json`; six figures shown inline and saved to `REPORTS/figures/`) | no | < 1 min |
| 6 | optional: `core.tools.eda report` (corpus / labels / protocol sections) | no | 1-2 min |
| 7 | **paste `REPORTS/nexar_eda_report.md` to Claude** (figures stay on Drive; the PNGs are not needed to read it) | no | - |

Thesis-V2 is only read; reports go to `Thesis-Final/outputs/nexar_eda/`. Upload branch `v_final_las_vad` to `Thesis-Final/kat-vad` first.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")


In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

DRIVE    = '/content/drive/MyDrive/Thesis'        # read-only
DRIVE_V2 = '/content/drive/MyDrive/Thesis-V2'     # read-only: Nexar_window
DRIVE_F  = '/content/drive/MyDrive/Thesis-Final'  # code, every output here
os.environ['PROJECT_ROOT']       = DRIVE_F
os.environ['KATVAD_DATA_ROOT']   = f'{DRIVE}/data'
os.environ['KATVAD_CACHE_ROOT']  = f'{DRIVE_F}/cache'
os.environ['KATVAD_CKPT_ROOT']   = f'{DRIVE}/ckpts'
os.environ['KATVAD_OUTPUT_ROOT'] = f'{DRIVE_F}/outputs'

REPO = Path(f'{DRIVE_F}/kat-vad')
assert (REPO / 'core' / 'constants.py').is_file(), f'no code at {REPO} -- upload branch v_final_las_vad there'
assert (REPO / 'core/tools/nexar_eda.py').is_file(), 'this upload has no nexar_eda -- re-upload kat-vad/'
assert '## 40. EDA of the Nexar training corpus' in (REPO / 'core/docs/final/KIM_RUN.md').read_text(), \
    'KIM_RUN.md has no §40 -- re-upload kat-vad/'
os.environ['PYTHONPATH'] = str(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
for name in [m for m in sys.modules if m == 'core' or m.startswith('core.')]:
    del sys.modules[name]              # constants reads KATVAD_* at import time
from core import constants  # noqa: E402

assert constants.REPO_ROOT == REPO, f'imported core from {constants.REPO_ROOT}, expected {REPO}'

WINDOW  = Path(DRIVE_V2) / 'data' / f'{constants.NEXAR_DATASET}_{constants.NEXAR_CONSTRUCTION_WINDOW}'   # K5f's corpus
ROOT    = constants.OUTPUT_ROOT / constants.NEXAR_EDA_OUTPUT_DIRNAME
REPORTS = ROOT / 'REPORTS'
LOCAL   = Path('/content/nexar_eda')
LOCAL.mkdir(parents=True, exist_ok=True)
assert WINDOW.is_dir(), f'no corpus at {WINDOW}'
print('corpus:', WINDOW, '| outputs:', ROOT)


## 3. Tests on the tool (CPU) - **raises if anything fails: stop and paste the output**

In [ ]:
ENV = dict(os.environ, USE_TF='0', TF_CPP_MIN_LOG_LEVEL='2')


def run(cmd):
    '''Stream a child's output; raise with its tail on failure.'''
    tail = []
    with subprocess.Popen([str(c) for c in cmd], cwd=str(REPO), env=ENV, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1) as proc:
        for line in proc.stdout:
            tail = (tail + [line])[-40:]
            print(line, end='')
    if proc.returncode:
        print('\n' + '=' * 30 + f'\nFAILED (exit {proc.returncode}) -- paste these lines to Claude:\n' + ''.join(tail))
        raise RuntimeError(f'exit {proc.returncode}: {" ".join(str(c) for c in cmd[:4])} ...')


run([sys.executable, '-m', 'pip', 'install', '-q', 'scikit-learn'])
run([sys.executable, '-m', 'pytest', '-q', '-p', 'no:cacheprovider', 'core/tests/test_nexar_eda.py'])


## 4. Corpus gate: this is K5f's corpus, copied to the VM once (the report reads three small files)

In [ ]:
local_data = LOCAL / 'data'
local_data.mkdir(exist_ok=True)
for f in (constants.LABELS_TRAIN_FILENAME, constants.META_FILENAME, constants.FRAME_LABELS_TEST_FILENAME,
          constants.WINDOWS_FILENAME, constants.DEFS_FILENAME):
    shutil.copy2(WINDOW / f, local_data / f)

labels = json.loads((local_data / constants.LABELS_TRAIN_FILENAME).read_text())
meta = json.loads((local_data / constants.META_FILENAME).read_text())
n_pos = sum(labels.values())
assert (len(labels), n_pos) == (7311, 1689), f"not K5f's corpus: {len(labels)} windows, {n_pos} positive"
sources = {str(meta[w]['source']) for w in labels}
assert all(meta[w]['source_is_abnormal'] for w in labels), 'a train window comes from a negative video (K5neg / K5fno, not K5f)'
print(f'Nexar_window: {n_pos} positive + {len(labels) - n_pos} normal windows from {len(sources)} positive videos')


## 5. The EDA (`core.tools.nexar_eda --plots`): refuses if the rebuilt labels differ from `meta.json`; figures under `REPORTS/figures/`

In [ ]:
from IPython.display import Image, display

out = LOCAL / 'reports'
run([sys.executable, '-m', 'core.tools.nexar_eda', '--data-dir', local_data, '--output-dir', out, '--plots'])
REPORTS.mkdir(parents=True, exist_ok=True)
stem = constants.NEXAR_EDA_REPORT_STEM
for ext in ('.json', '.md'):
    shutil.copy2(out / f'{stem}{ext}', REPORTS / f'{stem}{ext}')
figures = out / constants.NEXAR_EDA_FIGURES_DIRNAME
pngs = sorted(figures.glob('*.png'))
assert len(pngs) == 6, f'expected 6 figures, got {[p.name for p in pngs]}'
shutil.copytree(figures, REPORTS / constants.NEXAR_EDA_FIGURES_DIRNAME, dirs_exist_ok=True)
print((out / f'{stem}.md').read_text())
for png in pngs:       # dataset properties only: none of these is a model result
    print(png.name)
    display(Image(filename=str(png)))


## 6. Optional: the generic profiler on the same corpus (corpus / labels / protocol; no features, no probe)

In [ ]:
run([sys.executable, '-m', 'core.tools.eda', 'report', '--dataset', 'Nexar_window', '--data-dir', local_data,
     '--output-dir', LOCAL / 'eda', '--sections', 'corpus,labels,protocol', '--no-probe',
     '--mil-topk-pct', constants.DADA_ORIGIN_MIL_TOPK_PCT,    # K5f trained with loss.mil_topk_pct = 5 (not the default 16)
     '--score-head-kernel', constants.DADA_ORIGIN_SCORE_HEAD_KERNEL])  # and model.score_head_kernel = 3 (not the default 9)
shutil.copytree(LOCAL / 'eda', REPORTS / 'eda', dirs_exist_ok=True)
print((LOCAL / 'eda' / 'eda_report.md').read_text()[:6000])
drive.flush_and_unmount()          # C10: every pending Drive upload lands before the VM goes
print(f'Drive flushed -- paste {REPORTS}/{stem}.md to Claude')
